## Provenance and scope

This notebook is notebook 1 of the `cbase2/` pipeline (see `documentation.md`).
It ports **Steps 0--3** of the parent `Notebooks/AccountingConsistency.ipynb`
(ROADMAP §4.1): it loads the raw German 2019 use table, separates domestic from
imported uses, builds the conditional share matrices $\Omega^{D}$ (domestic) and
$\Omega^{raw}$ (total), decomposes value added into its four components, and
makes the one-factor aggregation assumption explicit.

**Inputs** (read-only): `data_raw/I-O_DE2019_formatiert.csv` (SHA-256 recorded
in `documentation.md`).

**Outputs** (written to `data_processed/`): `wrangling_omega_dom.csv`,
`wrangling_omega_raw.csv`, `wrangling_imports_taxes.csv`,
`wrangling_final_demand_taxes.csv`, `wrangling_value_added.csv`,
`wrangling_sector_shares.csv`, and the domestic intermediate matrix
`AC_domestic_intermediate_matrix.csv` (emitted here because $Z^{D}$ is built in
Step 1).

**Core principle** (§4.1): value added is kept as an explicit composite; the
notebook never relabels total value added as "labour". The proportional
allocation of imports across supplying sectors is a **stated assumption**
(the source table reports imports by *using* sector only).


## Step 0 -- Load and declare the schema

The sector/column layout is declared as **named constants** so the (non-obvious)
indexing is auditable rather than hidden in magic numbers. Sector rows are
$1{:}71$; the 71 sector columns sit at overall positions $2{:}72$ (column 1 is
the row-label column); the seven final-demand categories (private consumption,
private-organisations consumption, government consumption, equipment
investment, construction investment, inventories, exports) sit at positions
$75{:}81$. The import row ("Verwendung der Importe") and the product-tax row
are located **by name**, never by position.


In [ ]:
import Pkg

# Walk upward from this notebook to the nearest Project.toml (the BeyondHulten
# project root). Skipped when the running environment already provides the
# packages (e.g. the container stack environment used for headless validation).
function find_project(start)
    d = abspath(start)
    while d != dirname(d)
        isfile(joinpath(d, "Project.toml")) && return d
        d = dirname(d)
    end
    return nothing
end
if Base.find_package("CSV") === nothing
    Pkg.activate(find_project(@__DIR__))
end
using CSV, DataFrames, LinearAlgebra, Statistics, Plots

CBROOT = abspath(@__DIR__)                       # cbase2/ (this notebook's folder)
RAW    = joinpath(CBROOT, "data_raw", "I-O_DE2019_formatiert.csv")
OUT    = joinpath(CBROOT, "data_processed")      # this notebook's output stage
PLOTS  = joinpath(CBROOT, "plots")
mkpath(OUT); mkpath(PLOTS)
const datadir = joinpath(CBROOT, "data_raw")     # read-only inputs

# --- Declare the VERIFIED schema as named constants (auditable layout) ---
# (Indices confirmed by direct inspection of the raw CSV; see the parent
#  repo's docs/accounting_consistency_plan.md.)
N   = 71        # number of sectors
SEC = 2:72      # overall columns holding the 71 sectors (supplier rows / user cols)
FD  = 75:81     # overall columns: private cons, priv-orgs cons, gov cons,
                #   equipment inv, construction inv, inventories, exports

# Load the raw German 2019 use table (Destatis format: ';' delim, ',' decimal).
io = CSV.read(RAW, DataFrame; delim=';', decimal=',', missingstring=["-", "x"])
rename!(io, Symbol(names(io)[1]) => :Sektoren)
io.Sektoren = replace.(io.Sektoren, r"^\s+" => "")   # strip leading whitespace
io = coalesce.(io, 0)                                # missing ("-"/"x") -> 0

println("Loaded table: nrow=", nrow(io), " ncol=", ncol(io), " sectors=", N)
@assert nrow(io) >= N      "raw table has fewer rows than the declared schema"
@assert ncol(io) >= last(FD) "raw table has fewer columns than the declared schema"

# Helper: pull a named accounting row as a Float64 vector over a column range.
rowvec(name; cols=SEC) = Vector{Float64}(io[findfirst(==(name), io.Sektoren), cols])


## Step 1 -- Separate domestic vs imported uses; basic vs purchaser prices

Imports and product taxes are reported **by column** (using sector + final
category). Both rows are indexed DIRECTLY by their original column numbers
($SEC = 2{:}72$, $FD = 75{:}81$) -- slicing `2:end` first and re-indexing
afterwards shifts every import/tax value by one column (the off-by-one trap
documented in the parent pipeline; it silently produced a spurious residual in
earlier versions).

The **domestic** intermediate matrix $Z^{D}$ scales the raw matrix $Z$
(domestic + imported, by supplier $s$ and user $u$) by the domestic share of
each user's total intermediate input -- the proportional-allocation assumption
stated above:

$$\;Z^{D}_{s,u} = Z_{s,u} \cdot \frac{\text{domestic intermediate into } u}{\text{total intermediate into } u}\;$$

Users whose recorded imports exceed their total intermediate use are clamped to
a fully import-dependent domestic row (expected in a handful of sectors, not an
error). From $Z^{D}$ we form the user-by-supplier conditional share matrices,
each **row summing to 1** (orientation is asserted, not assumed):

- $\Omega^{D}$ -- domestic conditional shares (audit / incidence breakdown);
- $\Omega^{raw}$ -- total conditional shares (the model's equilibrium
  technology and `consumption_share` calibration use these).

Each user row sums to 1 -- **except** rows of fully import-dependent users
(zero domestic intermediate input), which are encoded explicitly as all-zero
("no domestic content") rather than left as the 0/0 = NaN the parent notebook
silently carried. The orientation and both row-sum conventions are asserted,
not assumed.


In [ ]:
# ---------------------------------------------------------------------------
# Step 1 -- Separate domestic vs imported uses; basic vs purchaser prices.
# ---------------------------------------------------------------------------
r_imp = findfirst(==("Verwendung der Importe"), io.Sektoren)
r_tx  = findfirst(==("Gütersteuern abzüglich Gütersubventionen"), io.Sektoren)
@assert r_imp !== nothing && r_tx !== nothing "import/product-tax rows not found by name"
imp_inter_byuser = Matrix{Float64}(io[r_imp:r_imp, SEC])[:]   # 1x71 slice -> flatten
imp_final        = Matrix{Float64}(io[r_imp:r_imp, FD])[:]
imptx_inter      = Matrix{Float64}(io[r_tx:r_tx, SEC])[:]
imptx_final      = Matrix{Float64}(io[r_tx:r_tx, FD])[:]

# Intermediate-use matrix Z[s,u]: value supplied by sector s to using sector u
# (domestic + imported combined, at the table's recorded price level).
Z = Matrix{Float64}(io[1:N, SEC])

imp_inter_total  = sum(imp_inter_byuser)
imp_final_total  = sum(imp_final)
imptx_final_total = sum(imptx_final)

# Total intermediate use into each user u, and its domestic part.
inter_into_user     = sum(Z; dims=1)[:]
dom_inter_into_user = max.(inter_into_user .- imp_inter_byuser, 0.0)  # clamp
n_clamped = count(inter_into_user .- imp_inter_byuser .< 0)
n_clamped > 0 && println("WARNING: domestic remainder clamped to 0 for ", n_clamped,
                         " fully import-dependent users (expected, not an error).")
denom = max.(inter_into_user, 1e-12)   # guard against zero intermediate use

# Domestic intermediate matrix: allocate imports proportionally across suppliers.
Z_dom = Z .* (dom_inter_into_user' ./ denom')
@assert isapprox(sum(Z_dom; dims=1)[:], dom_inter_into_user; rtol=1e-9)

# Conditional input-share matrices (user-by-supplier). Users whose domestic
# intermediate input is ZERO (fully import-dependent after the clamp above)
# have an undefined share row (0/0 = NaN); we encode those rows explicitly as
# all-zero -- the honest audit statement is "no domestic content". The parent
# notebook left them as NaN (it never asserted the row sums); here the
# treatment is explicit and asserted.
inter_dom_totals = sum(Z_dom; dims=1)[:]
zero_dom = vec(inter_dom_totals) .<= 0.0
Ω_dom = Z_dom' ./ vec(inter_dom_totals)     # Ω_dom[u,s]
Ω_raw = Z'    ./ vec(sum(Z; dims=1))        # Ω_raw[u,s]
zero_raw = vec(sum(Z; dims=1)) .<= 0.0
Ω_dom[zero_dom, :] .= 0.0
Ω_raw[zero_raw, :] .= 0.0
dom_ok = isapprox.(vec(sum(Ω_dom; dims=2)), 1.0; rtol=1e-9) .| zero_dom
raw_ok = isapprox.(vec(sum(Ω_raw; dims=2)), 1.0; rtol=1e-9) .| zero_raw
@assert all(dom_ok) "Ω_dom rows must sum to 1 (or 0 for fully import-dependent users)"
@assert all(raw_ok) "Ω_raw rows must sum to 1 (or 0 for users without intermediate input)"
count(zero_dom) > 0 && println("Ω_dom: ", count(zero_dom),
    " fully import-dependent users encoded as all-zero domestic share rows.")

println("Imported intermediate (total, EUR m)   = ", round(imp_inter_total))
println("Imported final demand   (total, EUR m) = ", round(imp_final_total))
println("Product taxes on final demand (total)  = ", round(imptx_final_total))
println("Product taxes on intermediate (total)  = ", round(sum(imptx_inter)))

# --- Persist wrangling intermediates for 02_accounting_consistency.ipynb ---
# Full Float64 precision (no rounding) so downstream computations are exact.
omega_raw_df = hcat(DataFrame(user = io.Sektoren[1:N]),
                    DataFrame(Ω_raw, :auto); makeunique=true)
rename!(omega_raw_df, ["user"; ["s$i" for i in 1:N]])
CSV.write(joinpath(OUT, "wrangling_omega_raw.csv"), omega_raw_df)
omega_dom_df = hcat(DataFrame(user = io.Sektoren[1:N]),
                    DataFrame(Ω_dom, :auto); makeunique=true)
rename!(omega_dom_df, ["user"; ["s$i" for i in 1:N]])
CSV.write(joinpath(OUT, "wrangling_omega_dom.csv"), omega_dom_df)

CSV.write(joinpath(OUT, "wrangling_imports_taxes.csv"),
    DataFrame(sector = io.Sektoren[1:N],
              imported_intermediate = imp_inter_byuser,
              product_tax_intermediate = imptx_inter))
CSV.write(joinpath(OUT, "wrangling_final_demand_taxes.csv"),
    DataFrame(category = ["private_consumption","private_orgs_consumption",
                          "government_consumption","equipment_investment",
                          "construction_investment","inventories","exports"],
              imported_final = imp_final,
              product_tax_final = imptx_final))

# The domestic intermediate matrix is emitted here (it is a Step 1 object);
# same format as the parent pipeline's artifact for comparability.
open(joinpath(OUT, "AC_domestic_intermediate_matrix.csv"), "w") do f
    write(f, "supplier\\user," * join(io.Sektoren[1:N], ",") * "\n")
    for s in 1:N
        write(f, io.Sektoren[s] * "," * join(round.(Z_dom[s, :]; digits=3), ",") * "\n")
    end
end
println("Wrangling intermediates written to data_processed/.")


## Step 2 -- Decompose value added

Instead of collapsing value added into one "labour" factor, we read its four
components and **assert they sum to gross value added** to machine precision.
This is the explicit decomposition required by §4.1:

- Compensation of employees (`Arbeitnehmerentgelt im Inland`)
- Other production taxes less subsidies (`Sonst.Produktionsabgaben abzgl. sonst.Subventionen`)
- Consumption of fixed capital (`Abschreibungen`)
- Net operating surplus (`Nettobetriebsüberschuss`)

Gross output at basic prices (`Produktionswert`) is recorded alongside.


In [ ]:
# ---------------------------------------------------------------------------
# Step 2 -- Decompose value added into its components (do NOT relabel as "labour").
# ---------------------------------------------------------------------------
gva     = rowvec("Bruttowertschöpfung")                        # gross value added (basic)
wage    = rowvec("Arbeitnehmerentgelt im Inland")              # compensation of employees
othertx = rowvec("Sonst.Produktionsabgaben abzgl. sonst.Subventionen")
dep     = rowvec("Abschreibungen")                             # consumption of fixed capital
netop   = rowvec("Nettobetriebsüberschuss")                    # net operating surplus
prodval = rowvec("Produktionswert")                           # gross output, basic prices

# Identity check: the four components must sum to gross value added.
va_total = wage .+ othertx .+ dep .+ netop
@assert isapprox(va_total, gva; rtol=1e-9) "VA decomposition must equal gross value added"

va_comp = DataFrame(sector = io.Sektoren[1:N],
                    wage = wage, other_production_tax = othertx,
                    depreciation = dep, net_operating_surplus = netop,
                    gross_value_added = gva)
CSV.write(joinpath(OUT, "wrangling_value_added.csv"), va_comp)

println("VA components sum to GVA: OK (max abs diff = ", maximum(abs.(va_total .- gva)), ")")
println("Mean wage share of GVA     = ", round(mean(wage ./ gva); digits=4))
println("Mean net-operating-surplus = ", round(mean(netop ./ gva); digits=4))
println("Mean other-production-tax  = ", round(mean(othertx ./ gva); digits=4))
println("Mean depreciation          = ", round(mean(dep ./ gva); digits=4))


## Step 3 -- Make the one-factor aggregation assumption explicit

The one-factor CGE treats **value added as a single composite primary factor**.
We keep that composite (`factor_share = gva / gross_output`) and *separately*
expose the labour-compensation share (`wage_share_gross_output`). The model's
historical `labor_share` field is thus a composite value-added weight, and the
narrative can no longer silently relabel total value added as "labour".


In [ ]:
# ---------------------------------------------------------------------------
# Step 3 -- Explicit one-factor aggregation assumption + share artifact.
# ---------------------------------------------------------------------------
factor_share  = gva ./ prodval     # VA share of gross output (basic) -- COMPOSITE factor
wage_share_go = wage ./ prodval    # labour-compensation share of gross output

sector_shares = DataFrame(sector = io.Sektoren[1:N],
                          gross_output_basic = prodval,
                          factor_share = factor_share,
                          wage_share_gross_output = wage_share_go)
CSV.write(joinpath(OUT, "wrangling_sector_shares.csv"), sector_shares)

@assert all(isfinite, factor_share)  && all(factor_share .> 0)  "factor shares must be positive"
@assert all(isfinite, wage_share_go) && all(wage_share_go .> 0) "wage shares must be positive"
@assert all(factor_share .<= 1.0 + 1e-9) "value added cannot exceed gross output"

println("Mean factor_share (VA / gross output)     = ", round(mean(factor_share); digits=4))
println("Mean wage_share (labour / gross output)   = ", round(mean(wage_share_go); digits=4))
println("(Note: 'factor_share' is the COMPOSITE VA weight, not pure labour.)")


## Diagnostic figure -- import intensity by sector

The imported-intermediate share of gross output motivates the financing-closure
design (F3 external debt absorbs demand through net imports mechanically; the
Armington hook is the $\Omega^{D}/\Omega^{raw}$ split). The figure is written to
`plots/01_import_intensity.png` and rendered inline next to the computation.


In [ ]:
# ---------------------------------------------------------------------------
# Diagnostic: imported-intermediate intensity by sector (share of gross output).
# ---------------------------------------------------------------------------
imp_share = imp_inter_byuser ./ max.(prodval, 1e-12)
p = bar(1:N, imp_share;
        label = "",
        title = "Imported intermediates as share of gross output (DE 2019)",
        xlabel = "sector", ylabel = "import share",
        size = (900, 400), margin = 4Plots.mm)
savefig(p, joinpath(PLOTS, "01_import_intensity.png"))
display(p)
println("Saved plots/01_import_intensity.png; unweighted mean import share = ",
        round(mean(imp_share); digits=4), "; max = ", round(maximum(imp_share); digits=4),
        " (sector ", argmax(imp_share), ")")


## Run summary

Artifacts written by this notebook (all under `data_processed/`), with the
final consistency gate re-stated. `02_accounting_consistency.ipynb` consumes
exactly these files plus the raw table.


In [ ]:
# ---------------------------------------------------------------------------
# Final gate for notebook 01: re-assert every invariant in one place and
# list the artifacts the next stage will read.
# ---------------------------------------------------------------------------
@assert isapprox(va_total, gva; rtol=1e-9)             "VA = GVA"
@assert isapprox(sum(Z_dom; dims=1)[:], dom_inter_into_user; rtol=1e-9) "Z_dom column sums"
@assert all(isapprox.(vec(sum(Ω_dom; dims=2)), 1.0; rtol=1e-9) .| zero_dom) "Ω_dom rows sum to 1 (or 0)"
@assert all(isapprox.(vec(sum(Ω_raw; dims=2)), 1.0; rtol=1e-9) .| zero_raw) "Ω_raw rows sum to 1 (or 0)"
println("All 01 assertions passed.")
for f in ["wrangling_omega_raw.csv", "wrangling_omega_dom.csv",
          "wrangling_imports_taxes.csv", "wrangling_final_demand_taxes.csv",
          "wrangling_value_added.csv", "wrangling_sector_shares.csv",
          "AC_domestic_intermediate_matrix.csv"]
    @assert isfile(joinpath(OUT, f)) "missing artifact: $f"
    println("  wrote data_processed/", f)
end
